In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')



Mounted at /content/drive


In [8]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git' --quiet
!pip install easyocr --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 12.5 MB/s eta 0:00:00


In [9]:
import os, cv2, torch, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from torchvision.ops import nms
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

import detectron2
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

import easyocr

In [10]:
COCO_OUT    = "/content/drive/MyDrive/Newspaper_Project/coco_dataset"
TRAIN_JSON  = f"{COCO_OUT}/annotations/train.json"
VAL_JSON    = f"{COCO_OUT}/annotations/val.json"
MODEL_PATH  = "/content/drive/MyDrive/Newspaper_Project/model_output/model_final.pth"
BERT_DIR    = "/content/drive/MyDrive/Newspaper_Project/news_classifier/final_model"

In [11]:
#load detectron
for name in ["newspaper_train", "newspaper_val"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances("newspaper_train", {}, TRAIN_JSON, image_root="")
register_coco_instances("newspaper_val",   {}, VAL_JSON,   image_root="")
MetadataCatalog.get("newspaper_train").thing_classes = ["article", "ads", "headline"]
MetadataCatalog.get("newspaper_val").thing_classes   = ["article", "ads", "headline"]

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES           = 3
cfg.MODEL.WEIGHTS                         = MODEL_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST     = 0.7
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST       = 0.3
cfg.TEST.DETECTIONS_PER_IMAGE             = 100

predictor = DefaultPredictor(cfg)
metadata  = MetadataCatalog.get("newspaper_val")
print(" Detectron2 loaded")

 Detectron2 loaded


In [12]:
#Load EasyOCR + DistilBERT ──
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

reader     = easyocr.Reader(['en'], gpu=(DEVICE == "cuda"))
tokenizer  = DistilBertTokenizerFast.from_pretrained(BERT_DIR)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_DIR).to(DEVICE)
bert_model.eval()

LABEL_MAP  = {0: "article", 1: "ads", 2: "headline"}  # detectron2 0-indexed
CATEGORIES = [
    "Politics", "Business & Economy", "Sports", "Entertainment",
    "Science & Technology", "Health & Medicine", "Crime & Law",
    "Education", "Environment", "Lifestyle", "Travel",
    "Religion & Culture", "Parenting", "Opinion & Editorial", "International"
]
print(f"✓ EasyOCR + DistilBERT loaded on {DEVICE}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✓ EasyOCR + DistilBERT loaded on cuda


In [13]:
#Pipeline Functions
def clean_instances(instances, iou_threshold=0.3):
    if len(instances) == 0:
        return instances
    boxes, scores, classes = instances.pred_boxes.tensor, instances.scores, instances.pred_classes
    keep = []
    for cls_id in classes.unique():
        mask = (classes == cls_id)
        idx  = nms(boxes[mask], scores[mask], iou_threshold)
        keep.extend(mask.nonzero(as_tuple=True)[0][idx].tolist())
    return instances[sorted(set(keep))]

def ocr_crop(img_bgr, box, padding=5):
    h, w   = img_bgr.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1-padding), max(0, y1-padding)
    x2, y2 = min(w, x2+padding), min(h, y2+padding)
    crop   = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return ""
    return " ".join(reader.readtext(crop, detail=0, paragraph=True)).strip()

def classify_text(text):
    if not text or len(text) < 15:
        return None, 0.0
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding="max_length", max_length=128).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(bert_model(**inputs).logits, dim=1).squeeze()
    conf, pred = probs.max(0)
    return CATEGORIES[pred.item()], round(conf.item(), 3)

def analyze_newspaper(image_path):
    img_bgr   = cv2.imread(image_path)
    instances = clean_instances(predictor(img_bgr)["instances"].to("cpu"))
    results   = []
    for box, cls_id, score in zip(instances.pred_boxes.tensor.numpy(),
                                   instances.pred_classes.numpy(),
                                   instances.scores.numpy()):
        rtype    = LABEL_MAP[int(cls_id)]
        text     = ocr_crop(img_bgr, box)
        category, conf = classify_text(text) if rtype == "article" else (None, 0.0)
        results.append({
            "box": box.tolist(), "type": rtype,
            "score": round(float(score), 3),
            "text": text[:200], "category": category, "confidence": conf
        })
    return results

In [16]:
# ── CELL 7: Run + Visualize ──
def visualize(image_path, results):
    img = Image.open(image_path).convert("RGB")
    fig, ax = plt.subplots(figsize=(14, 18))
    ax.imshow(img)
    COLOR = {"article": "blue", "headline": "green", "ads": "red"}
    for r in results:
        x1, y1, x2, y2 = r["box"]
        c = COLOR[r["type"]]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor=c, facecolor="none"))
        label = r["type"]
        if r["category"]:
            label += f"\n{r['category']} ({r['confidence']:.0%})"
        ax.text(x1, y1-4, label, fontsize=6, color="white",
                bbox=dict(facecolor=c, alpha=0.7, pad=1))
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Pick a random val image and run
dataset_dicts = DatasetCatalog.get("newspaper_val")
sample        = random.choice(dataset_dicts)
image_path    = sample["file_name"]

print(f"Image: {image_path.split('/')[-1]}\n")
results = analyze_newspaper(image_path)

print(f"{'TYPE':<10} {'CATEGORY':<25} {'CONF':<6} TEXT PREVIEW")
print("-" * 70)
for r in results:
    print(f"{r['type']:<10} {str(r['category']):<25} {r['confidence']:<6} {r['text'][:40]!r}")

visualize(image_path, results)

Output hidden; open in https://colab.research.google.com to view.